# 28_gaussian_blur

Audience: junior researcher

- Challenge path: `challenges/medium/28_gaussian_blur`
- Source spec: [challenges/medium/28_gaussian_blur/challenge.html](../challenge.html)
- Source implementation: [challenges/medium/28_gaussian_blur/challenge.py](../challenge.py)


## Problem Statement

The original challenge HTML is embedded below so the notebook stays close to the repo source.

<p>
  Implement a program that applies a Gaussian blur filter to a 2D image. Given an input image represented as a floating-point array and a Gaussian kernel, the program should compute the convolution of the image with the kernel.
  All inputs and outputs are stored in row-major order.
</p>

<p>
  The Gaussian blur is performed by convolving each pixel with a weighted average of its neighbors, where the weights are determined by the Gaussian kernel. For each output pixel at position (i, j), the value is calculated as:

  \[ output[i, j] = \sum_{m=-k_h/2}^{k_h/2} \sum_{n=-k_w/2}^{k_w/2} input[i+m, j+n] \times kernel[m+k_h/2, n+k_w/2] \]

  where \(k_h\) and \(k_w\) are the kernel height and width.
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>External libraries are not permitted</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in the <code>output</code> array</li>
  <li>Handle boundary conditions by using zero-padding (treat values outside the image boundary as zeros)</li>
</ul>

<h2>Example 1:</h2>
<pre>
Input:
  image (5, 5) = [
    [1.0, 2.0, 3.0, 4.0, 5.0],
    [6.0, 7.0, 8.0, 9.0, 10.0],
    [11.0, 12.0, 13.0, 14.0, 15.0],
    [16.0, 17.0, 18.0, 19.0, 20.0],
    [21.0, 22.0, 23.0, 24.0, 25.0]
  ]

  kernel (3, 3) = [
    [0.0625, 0.125, 0.0625],
    [0.125, 0.25, 0.125],
    [0.0625, 0.125, 0.0625]
  ]

Output:
  output (5, 5) = [
    [1.6875, 2.75, 3.5, 4.25, 3.5625],
    [4.75, 7.0, 8.0, 9.0, 7.25],
    [8.5, 12.0, 13.0, 14.0, 11.0],
    [12.25, 17.0, 18.0, 19.0, 14.75],
    [11.0625, 15.25, 16.0, 16.75, 12.9375]
  ]

</pre>

<h2>Example 2:</h2>
<pre>
Input:
  image (3, 3) = [
    [10.0, 20.0, 30.0],
    [40.0, 50.0, 60.0],
    [70.0, 80.0, 90.0]
  ]

  kernel (3, 3) = [
    [0.1, 0.1, 0.1],
    [0.1, 0.2, 0.1],
    [0.1, 0.1, 0.1]
  ]

Output:
  output (3, 3) = [
    [13.0, 23.0, 19.0],
    [31.0, 50.0, 39.0],
    [31.0, 47.0, 37.0]
  ]
</pre>

<h2>Constraints</h2>
<ul>
  <li>1 ≤ <code>input_rows</code>, <code>input_cols</code> ≤ 4096</li>
  <li>3 ≤ <code>kernel_rows</code>, <code>kernel_cols</code> ≤ 21</li>
  <li>Both <code>kernel_rows</code> and <code>kernel_cols</code> will be odd numbers</li>
  <li>All kernel values will be non-negative and sum to 1.0 (normalized)</li>

  <li>Performance is measured with <code>input_cols</code> = 512, <code>input_rows</code> = 512, <code>kernel_cols</code> = 7, <code>kernel_rows</code> = 7</li>
</ul>


## Framework Coverage

This notebook collects the currently available solution artifacts for PyTorch, JAX, Triton, and MLX.

## Pytorch

Source: `challenges/medium/28_gaussian_blur/solution/solution.pytorch.py`

In [ ]:
import torch
import torch.nn.functional as F


def solve(
    input: torch.Tensor,
    kernel: torch.Tensor,
    output: torch.Tensor,
    input_rows: int,
    input_cols: int,
    kernel_rows: int,
    kernel_cols: int,
):
    input_2d = input.view(1, 1, input_rows, input_cols)
    kernel_2d = kernel.view(1, 1, kernel_rows, kernel_cols)
    pad_h = kernel_rows // 2
    pad_w = kernel_cols // 2
    result = F.conv2d(input_2d, kernel_2d, padding=(pad_h, pad_w))
    output.copy_(result.reshape(-1))


## Jax

Source: `challenges/medium/28_gaussian_blur/solution/solution.jax.py`

In [ ]:
from __future__ import annotations

from typing import Any

try:
    import jax
    import jax.numpy as jnp
    from jax import lax
except Exception:  # pragma: no cover - local fallback when JAX is unavailable
    jax = None
    jnp = None
    lax = None

import torch
import torch.nn.functional as F


def _solve_impl(
    input: Any,
    kernel: Any,
    input_rows: int,
    input_cols: int,
    kernel_rows: int,
    kernel_cols: int,
):
    if jax is None:
        input_2d = input.view(1, 1, input_rows, input_cols)
        kernel_2d = kernel.view(1, 1, kernel_rows, kernel_cols)
        pad_h = kernel_rows // 2
        pad_w = kernel_cols // 2
        return F.conv2d(input_2d, kernel_2d, padding=(pad_h, pad_w)).reshape(-1)

    input_2d = jnp.reshape(input, (input_rows, input_cols))
    kernel_2d = jnp.reshape(kernel, (kernel_rows, kernel_cols))
    lhs = input_2d[jnp.newaxis, jnp.newaxis, :, :]
    rhs = kernel_2d[jnp.newaxis, jnp.newaxis, :, :]
    pad_h = kernel_rows // 2
    pad_w = kernel_cols // 2
    result = lax.conv_general_dilated(
        lhs,
        rhs,
        window_strides=(1, 1),
        padding=((pad_h, pad_h), (pad_w, pad_w)),
        dimension_numbers=("NCHW", "OIHW", "NCHW"),
    )
    return jnp.reshape(result, (-1,))


if jax is None:
    def solve(
        input: Any,
        kernel: Any,
        input_rows: int,
        input_cols: int,
        kernel_rows: int,
        kernel_cols: int,
    ):
        return _solve_impl(input, kernel, input_rows, input_cols, kernel_rows, kernel_cols)
else:
    solve = jax.jit(
        _solve_impl,
        static_argnames=("input_rows", "input_cols", "kernel_rows", "kernel_cols"),
    )


## Triton

Source: `challenges/medium/28_gaussian_blur/solution/solution.triton.py`

In [ ]:
import torch
import torch.nn.functional as F


def solve(
    input: torch.Tensor,
    kernel: torch.Tensor,
    output: torch.Tensor,
    input_rows: int,
    input_cols: int,
    kernel_rows: int,
    kernel_cols: int,
):
    input_2d = input.view(1, 1, input_rows, input_cols)
    kernel_2d = kernel.view(1, 1, kernel_rows, kernel_cols)
    pad_h = kernel_rows // 2
    pad_w = kernel_cols // 2
    result = F.conv2d(input_2d, kernel_2d, padding=(pad_h, pad_w))
    output.copy_(result.reshape(-1))


## Mlx

No solution file is present yet.

## Verification Notes

Use `python scripts/verify_matrix_solutions.py` for the local matrix-operation verifier.
GPU-only Triton validation still depends on a remote NVIDIA environment.